# v122_full_stack — v110 + every switch merged after it, on the tight mock

| Field | Value |
|---|---|
| **Version** | `v122_full_stack` |
| **Plan group** | INT (integration of switches from plan groups A1, B2–B4, C, D3 and E) |
| **Parent version** | `v110` (`v110_m3_features`: est_public 0.97313, public 0.966) |
| **Author** | M3 |
| **Date** | 2026-09-26 |
| **Status** | running (background script `run_v122.py`; it logs KEEP or DROP at its end) |
| **Hypothesis** | Every opt-in switch merged after v110 (rules v5, learned filler words, learned token evidence, stage 1 in 2 bags, stage-2 extra groups, stage 2 on fit + tune entities with 127 leaves and 3 seeds), turned on together, lifts est_public on the tight mock above v110's 0.97313. |

v110 is the best benchmark so far (est_public 0.97313; public 0.966): v107's two-stage matcher
with dense blocking (B6 + the name + address-number exact pass, `db1f33c4`), a GPU stage 1
trained on train-fold entities absent from the mock, M3's four feature groups in stage 1, rival
features in stage 2 and v107's rule tuning on the tight mock. Everything merged after it was
built behind an opt-in switch and tested on its own, but never run together on the mock. v122
keeps v110's recipe and turns every switch on:

* **normalisation rules v5**: French forms, `+` / `et` as `and`, leet legal forms, the record's
  own country word out of the core name;
* **learned filler words**: the filler-free exact blocking pass (`nofill_max_group=50`) and the
  `nofill` feature group;
* **learned token evidence**: the `tok_evidence` group, in stage 1 so the candidate filter reads
  it too;
* **stage 1 in 2 bags** over the training entities instead of one GPU-capped model;
* **stage-2 extras** on the kept pairs: `interactions` + `missing_flags` (M3, v044: +0.0002) and
  `phonetic` (M2, PR #10);
* **stage 2 on fit + tune entities** (out of fold), **127 leaves**, **3 seeds** averaged.

The experiment is the script `run_v122.py` in this folder, not this notebook: it runs for hours
in the background and caches every step (normalisation, blocking per tag, stage-1 chunks and
model, stage-1 outputs per country, stage-2 seed models, test stage-1 outputs), so a re-run
resumes where it stopped. It ends with `tracking.log_result` (`metrics.json` here, the `v122`
row of `experiments/experiments.csv`) and, for a KEEP, the test files. **This notebook
recomputes nothing**: its cells read `metrics.json`, the parent's `metrics.json`, `run.log` and
`artifacts/two_stage/rule.json`, and print "run not finished" until the script has logged its
result.

```
train-fold S1 absent from the mock (v110's sample: 212,941 entities)
    ─► rules v5 + learned token map + learned fillers ─► blocking db1f33c4 + filler-free pass
    ─► 94 features (v110's 76 + nofill 6 + tok_evidence 12) → disk chunks
    ─► XGBoost on the GPU, 2 bags on disjoint entity slices, averaged ─► stage 1
mock fold ─► stage 1 on every pair ─► filter (p1 ≥ 0.01, top 16) + competition, anchor, rival
    + extras on the kept pairs (interactions, missing_flags, phonetic)
    ─► stage 2: XGBoost on the GPU, 127 leaves, cross-fitted on fit + tune entities,
       seeds 42 / 43 / 44 averaged per part
    ─► v107's rule choice on mock tune ─► est_public on mock val ─► KEEP if > 0.97313
    ─► KEEP only: test files in submissions/v122/ and output/
```

## 1. Hypothesis

* **Change vs parent (v110):** the six switches above, each opt-in (its default is v110's
  behaviour) and merged after v110; details in §4:
  1. normalisation rules v5 (v110 ran on v3; v4 was France-only);
  2. learned filler words: filler-free core name, exact blocking pass and `nofill` group;
  3. learned token evidence: the `tok_evidence` group in stage 1;
  4. stage 1 in 2 bags (`fit_stage1(bags=2)`);
  5. stage-2 extras `interactions`, `missing_flags` and `phonetic` on the kept pairs;
  6. stage 2 on fit + tune entities, 127 leaves (v110: 63), mean of seeds 42, 43 and 44.
* **Why it should raise est_public:** (1)–(3) target errors v110 still makes: true pairs whose
  names differ only by words the pool adds (removing the fillers makes +1.2 % of US and +2.6 %
  of Indian true pairs name-equal, TRACKER #74), and false merges of the "S1 name + decoy word"
  kind (`holdings`, `midtown`), which token-set similarity scores as matches (#75). (4) gives
  stage 1 more training rows than v110's 6.99M of 14.9M. (5) and (6) give stage 2 more signal,
  about 1.5× the entities, more capacity and steadier probabilities for the rule to cut.
* **Checks (all logged in `metrics.json`):** est_public against v110's 0.97313; stage 1 alone
  (`single_stage_est_public`) against v110's 0.96813; one seed against three
  (`one_seed_est_public`); candidate recall after the filter against v110's 0.9788; the extras'
  share of stage-2 gain; error counts against v110's.
* **Decision (the user's rule):** **KEEP if est_public > v110's 0.97313, else DROP** (no
  margin). A KEEP writes the test files to `submissions/v122/` and `output/`.
* **Caveat:** the tight mock predicted v107's public score to within 0.0001 but missed v110's
  (est 0.9731, public 0.966, flat on v107; submission 06, v121, tests whether France caused
  it). The mock has no French entity, so est_public cannot see the French half of rules v4 / v5.

## 2. Setup

Only `entity_resolution.config` is imported, for the paths: nothing here loads data or a model.
`read_json` returns None for a file the script has not written yet (or is writing), and
`finished()` guards every cell that needs the logged result. `log_entries` splits the run log
into its timestamped entries (`run_v122.log()` prints `[HH:MM:SS +  N.N min] message`; a table
printed with a message continues on the lines below it), so a cell can quote what the script
logged about one step while the run is still going.

In [ ]:
import ast
import json
import re
from itertools import pairwise
from pathlib import Path

import pandas as pd
from IPython.display import display

from entity_resolution import config as C

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 30)

VERSION = "v122_full_stack"
EXP_DIR = C.EXPERIMENTS / VERSION
PARENT_DIR = C.EXPERIMENTS / "v110_m3_features"
ARTIFACTS = EXP_DIR / "artifacts"                  # gitignored: models, rule, scored pairs
RUN_LOG = EXP_DIR / "run.log"                      # stdout + stderr of run_v122.py
RULE_JSON = ARTIFACTS / "two_stage" / "rule.json"
# the head of a log entry as run_v122.log() writes it: "[22:26:54 +   0.0 min] "
ENTRY = re.compile(r"^\[\d{2}:\d{2}:\d{2} \+ *[\d.]+ min\] ", re.MULTILINE)


def read_json(path: Path) -> dict | None:
    """The JSON object in ``path``; None while the script has not written it (or is writing)."""
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return None


def read_log() -> str | None:
    """The run log as text (UTF-8, or UTF-16 when a shell redirect wrote a BOM); None if absent."""
    if not RUN_LOG.exists():
        return None
    raw = RUN_LOG.read_bytes()
    utf16 = raw[:2] in (b"\xff\xfe", b"\xfe\xff")
    return raw.decode("utf-16" if utf16 else "utf-8", errors="replace")


def log_entries(*needles: str) -> list[str]:
    """Run-log entries (a timestamped line and its continuation lines) holding any needle."""
    text = read_log() or ""
    starts = [m.start() for m in ENTRY.finditer(text)] + [len(text)]
    entries = [text[a:b].rstrip() for a, b in pairwise(starts)]
    return [e for e in entries if any(n in e for n in needles)]


def finished() -> bool:
    """Whether the script has logged its result; prints "run not finished" while it has not."""
    if run is None:
        print("run not finished: run_v122.py writes metrics.json at its end "
              "(section 9 shows the run log so far)")
    return run is not None


run = read_json(EXP_DIR / "metrics.json")       # the experiments.csv row + "metrics"
parent = read_json(PARENT_DIR / "metrics.json")
v122 = run["metrics"] if run else {}
v110 = parent["metrics"] if parent else {}
# the script compares with the parent's unrounded est_public (0.97313 rounded)
EST_TO_BEAT = float(v122.get("parent_est_public", v110.get("est_public", 0.97313)))
print(f"v110 est_public to beat {EST_TO_BEAT:.5f}; run finished: {run is not None}"
      + (f"; decision {run.get('decision')}" if run else ""))
for entry in log_entries("est_public to beat"):     # the script's first line: its setup
    print(entry)

### Parameters of this version

The script holds them: `configs()` (v110's configuration with every later switch on), the
stage-1 parameters, the stage-1 sample size, the bags and the stage-2 seeds. They are printed
verbatim from `run_v122.py`, parsed with `ast` rather than imported (importing the script would
load the whole pipeline), so this notebook shows exactly what ran.

In [ ]:
SCRIPT = EXP_DIR / "run_v122.py"
SHOW = {"M3_GROUPS", "STAGE2_EXTRAS", "N_STAGE1", "STAGE1_BAGS", "STAGE2_SEEDS", "configs",
        "S1_PARAMS"}
script = SCRIPT.read_text(encoding="utf-8")
for node in ast.parse(script).body:
    if isinstance(node, ast.Assign):
        names = {t.id for t in node.targets if isinstance(t, ast.Name)}
    else:
        names = {getattr(node, "name", "")}
    if names & SHOW:
        print(ast.get_source_segment(script, node), end="\n\n")

## 3. Data

The mock fold, built by the script exactly as in v103–v110 (`build_mock` on the train and val
folds with `target_shape()`): per country the test's pool size and pool records per S1, and
1.37M present S1 entities with roles `fit` (693k), `tune` (340k) and `val` (340k: India 176,522,
US 163,562). Stage 1 trains on train-fold S1 entities absent from the mock, v110's id-hash
sample (212,941 entities), blocked against the train-fold pool. The cell quotes what the script
logged about both, and about the learned token map, filler words and token evidence.

In [ ]:
entries = log_entries("mock built", "token map", "stage 1:", "stage-1 set")
print("\n".join(entries) if entries else "nothing logged about the data yet")

## 4. Method

One subsection per switch: what it does and why. The cells show what the script recorded
(configuration, fit information, the chosen rule), next to v110's values.

### 4.1 Normalisation: rules v5 and learned filler words

Rules v5 (`normalize.RULES_VERSION = 5`; v110 ran on v3):

* v4, France-only (from the audit of French test records, TRACKER #58 / #59; US and Indian
  test records stay byte-identical to v3): the pool's `N°` / `Nº` number
  marker leaves addresses before folding (it became a stray `n` token), and `Compagnie` is a
  legal form like `Cie`;
* the pool's `&` written `et` (France) or as a standalone `+` (every country) reads as `and`;
  `Frs` reads as `Freres`;
* legal forms written in leet (`5ARL`, `C0rp`) are legal forms;
* French address abbreviations read like the full forms: `12B` (`12 bis`), `Crs`, `Psg` /
  `Pass`, `Appt` / `App`;
* the record's own country name, which Source 1 writes into names (`(France)`, `(India)`) and
  the pool drops, leaves `name_core`. Country selects no code path: the rule drops whatever the
  record's own country value is, so France needs nothing special.

Unlike v4, v5 changes US and Indian records too, so the whole pipeline runs on newly normalised
text.

Learned filler words (`NormaliseConfig(learn_fillers=True)`, `normalize.fit_fillers`): words the
pool writes into true matches' names that Source 1 lacks, learned from the train fold's true
pairs (Latin script). A word is a filler when the pool adds it at least as often as S1 names
genuinely hold it: `center`, `services`, `service`, alias markers (`dba`, `fka`, `formerly`),
`id` / `www` tails and misspelt legal forms (`lnc`). Words the pool adds to decoys (`holdings`,
`group`, `midtown`: other businesses) are held by S1 names but hardly ever added on true pairs,
so they are not learned. The list travels with stage 1 (`Fitted.fillers`), so test uses the
train fold's list; `name_core_nofill` (the core name without the fillers) is added on load and
the static columns never change.

### 4.2 Blocking and features

* **Blocking:** v110's `db1f33c4` (exact core, sorted and squashed name, pool groups ≤ 200; core
  name + a shared address number, ≤ 100; TF-IDF top-k on core-name character 3-grams (10),
  name + address words (50) and address words (10); at most 120 per S1, best cosine first) plus
  one exact pass on the sorted words of `name_core_nofill` (pool groups ≤ 50), so the pool's
  `Acme Services` meets S1 `Acme`. The run log's first line gives the new key (`8e915447`). The
  stage-1 filter is v104's: p1 ≥ 0.01 and at most 16 candidates per S1.
* **Stage-1 features (94):** v110's 76 (v101's 53 + M3's `idf`, `token_freq`, `ctx_idf`,
  `address_extra`), plus
  * `nofill` (6): ratio, token-set and Jaccard similarity of the filler-free core names,
    whether they hold the same words, and the fillers removed on each side;
  * `tok_evidence` (12, `evidence.py`): per side (words only the pool name holds, words only the
    S1 name holds), the sum, min and max of the learned log-odds and the counts of strong decoy
    words, strong filler words and unknown words. The log-odds come from near-duplicate pairs of
    the train fold whose word sets differ by exactly one word (true pairs are matches, the rest
    decoys); words never seen in train (French ones) count as unknown.
* **Stage-2 frame:** the 94 stage-1 features, v110's 20 columns derived from p1 (competition,
  anchor, rival) and 11 extras computed on the kept pairs only (~4.4 per S1 instead of ~69
  candidates):
  * `interactions` (3): strong name + weak address, strong address + weak name, both strong
    (the decoy and rename signatures, 07 §3);
  * `missing_flags` (2): no address number on the S1 / the pool side (the numeric scores are NaN
    then, 07 §1);
  * `phonetic` (6): exact match, Jaccard and S1-side overlap of the Soundex + Double Metaphone
    codes of the core names and of the addresses; NaN where either side was written in a
    non-Latin script (the codes of transliterated text are noise).

The configuration the script logged, next to v110's: rules, blocking key, feature groups,
stage-1 bags and the stage-2 settings. v110's `metrics.json` predates `rules_version` (v110 ran
on rules v3) and `stage2_seeds` (one model per part, seed 42).

In [ ]:
def settings(rec: dict, rules_default: int | None = None) -> dict:
    """The settings v122 changes, as one version's metrics.json records them."""
    tcfg = rec.get("two_stage", {})
    model = tcfg.get("model", {})
    return {
        "rules version": rec.get("rules_version", rules_default),
        "blocking key": rec.get("blocking_key"),
        "stage-1 feature groups": len(rec.get("feature_groups", [])),
        "stage-1 bags": (rec.get("stage1_fit_info") or {}).get("bags", 1),
        "stage-2 train roles": ", ".join(tcfg.get("train_roles", [])),
        "stage-2 extra groups": ", ".join(tcfg.get("extra_groups", [])) or "-",
        "stage-2 leaves": model.get("num_leaves"),
        "stage-2 learning rate": model.get("learning_rate"),
        "stage-2 seeds": ", ".join(str(s) for s in rec.get("stage2_seeds", [model.get("seed")])),
        "rivals / anchors / cohesion": " / ".join(str(tcfg.get(k))
                                                  for k in ("rivals", "anchors", "cohesion")),
    }


if finished():
    display(pd.DataFrame({"v110": settings(v110, rules_default=3),   # v110 predates the field
                          "v122": settings(v122)}))
    added = [g for g in v122.get("feature_groups", []) if g not in v110.get("feature_groups", [])]
    print("stage-1 feature groups added:", added)

### 4.3 Stage 1 in 2 bags

v110 trained one XGBoost model on the GPU on 6.99M of its 14.9M training rows: a 4 GB card
holds about 530M cells (`stage1.GPU_CELLS`), 7.0M rows × 76 features. `fit_stage1(bags=2)`
splits the training entities by id hash into two disjoint slices, trains one model per slice
(one after the other, each under the same cap: 530M / 94 features ≈ 5.6M rows) and averages
them (`SeedMean`), so the two bags see up to ≈ 11.3M rows; `entity_share_used` says how much of
the set they used (v110: 0.49). Early stopping reads the same held-out 5 % id slice for both
bags; `tune_logloss` / `tune_auc` are those of the averaged probability. The parameters are
v110's: 127 leaves, learning rate 0.1, early stopping after 50 rounds.

In [ ]:
KEYS = ["rows", "rows_per_bag", "entity_share_used", "best_iteration", "positive_rate",
        "tune_logloss", "tune_auc", "fit_seconds"]
if finished():
    fi122, fi110 = v122.get("stage1_fit_info") or {}, v110.get("stage1_fit_info") or {}
    display(pd.DataFrame({  # object columns: every value as logged (ints, lists, None)
        "v110": pd.Series({k: fi110.get(k) for k in KEYS}, dtype=object),
        "v122": pd.Series({k: fi122.get(k) for k in KEYS}, dtype=object)}))
for entry in log_entries("stage-1 fit"):            # logged as soon as stage 1 is trained
    print(entry)

### 4.4 Stage 2: fit + tune entities, 127 leaves, 3 seeds

v110 trained stage 2 on the mock's fit entities only (693k), cross-fitted in two parts, 63
leaves, one seed, with early stopping on tune entities. v122 trains on fit + tune entities
(1.03M, `train_roles=("fit", "tune")`); each is scored by the part model that never saw it, so
the rule tuned on the tune entities still sees out-of-fold probabilities, and early stopping
reads each model's held-out part instead of tune entities. 127 leaves at learning rate 0.05;
the cross-fitted pair of models is trained once per seed (42, 43, 44) and the seeds are
averaged per part (`SeedMean`). Each seed's models are saved as soon as they are fitted, so a
re-run loads them. The table: per seed and part, the training rows, the best iteration and the
held-out log-loss and AUC.

In [ ]:
if finished():
    parts = [{"seed": seed, "part": part, **info}
             for seed, seed_info in (v122.get("stage2_fit_info") or {}).items()
             for part, info in seed_info.items() if isinstance(info, dict)]
    if parts:
        display(pd.DataFrame(parts).set_index(["seed", "part"]))
    else:
        print("no stage-2 fit info in metrics.json")
for entry in log_entries("stage 2 seed"):           # one line per seed as it finishes
    print(entry)

### 4.5 Decision rule

v107's rule choice, as in v110: each pool record stays only with its highest-probability S1
(1-to-1 across every present entity), then two rule families are tuned on the mock's tune
entities for the tight score (false merges ×1.45): the threshold grid (`tune_mock`) and
expected-F0.5 decoding (`tune_expected`: pair probability p^γ, m expected misses). The family
with the higher tune score gives the rule. v110 chose expected-F0.5 decoding (γ 1.5, m 0.1, at
most 11 matches). `threshold_rule` is the best threshold rule, logged either way; `rule.json`,
saved with the two-stage model, adds the chosen rule's tune score.

In [ ]:
saved_rule = read_json(RULE_JSON)
if finished():
    display(pd.DataFrame({
        f"v110 ({v110.get('rule_kind')})": pd.Series(v110.get("rule", {}), dtype=object),
        f"v122 ({v122.get('rule_kind')})": pd.Series(v122.get("rule", {}), dtype=object),
        "v122 best threshold rule": pd.Series(v122.get("threshold_rule", {}), dtype=object),
    }))
print("artifacts/two_stage/rule.json:", json.dumps(saved_rule) if saved_rule else "not written yet")

## 5. Evaluation on the mock

The primary number is **est_public** on the mock's val entities (340,084 S1: India 176,522, US
163,562): the tight score (false merges weighted ×1.45) minus 0.0072, calibrated on the
leaderboard. Plain mock F0.5, singletons, pair precision and recall are diagnostics. Deltas are
against v110's logged `metrics.json`, which names plain mock F0.5 `mock_f_beta` and stage 1
alone's `single_stage_mock_f_beta`.

### 5.1 v122 against v110

In [ ]:
ROWS = {  # label: (v122 key, v110 key)
    "est_public": ("est_public", "est_public"),
    "f_tight": ("f_tight", "f_tight"),
    "mock F0.5": ("f_beta", "mock_f_beta"),
    "F0.5 singletons": ("f_beta_singletons", "f_beta_singletons"),
    "F0.5 matched": ("f_beta_matched", "f_beta_matched"),
    "pair precision": ("pair_precision", "pair_precision"),
    "pair recall": ("pair_recall", "pair_recall"),
    "stage 1 alone: est_public": ("single_stage_est_public", "single_stage_est_public"),
    "stage 1 alone: mock F0.5": ("single_stage_f_beta", "single_stage_mock_f_beta"),
    "candidate recall (val, after the filter)": ("cand_recall_val", "cand_recall_val"),
    "candidates per S1 (val)": ("cands_mean_val", "cands_mean_val"),
}
if finished():
    cmp = pd.DataFrame({label: {"v110": v110.get(k110), "v122": v122.get(k122)}
                        for label, (k122, k110) in ROWS.items()}).T.astype(float)
    cmp["delta"] = cmp["v122"] - cmp["v110"]
    display(cmp.round(5))
    est = float(v122.get("est_public", float("nan")))
    print(f"est_public {est:.5f} vs v110 {EST_TO_BEAT:.5f} ({est - EST_TO_BEAT:+.5f}): "
          f"{'KEEP' if est > EST_TO_BEAT else 'DROP'} by the rule; logged {run.get('decision')}")

### 5.2 By country

v110's `metrics.json` holds mock F0.5 per country but not est_public per country; its notebook
printed them (§4.3: India 0.97165, US 0.97473), which fill the gap.

In [ ]:
V110_EST_BY_COUNTRY = {"India": 0.971648, "US": 0.974731}   # v110 notebook, section 4.3
if finished():
    countries = sorted(k.removeprefix("est_public_") for k in v122 if k.startswith("est_public_"))
    if countries:
        by_country = pd.DataFrame({c: {
            "est_public v110": v110.get(f"est_public_{c}", V110_EST_BY_COUNTRY.get(c)),
            "est_public v122": v122.get(f"est_public_{c}"),
            "mock F0.5 v110": v110.get(f"mock_{c}"),
            "mock F0.5 v122": v122.get(f"mock_{c}"),
        } for c in countries}).T.astype(float)
        by_country["d_est_public"] = by_country["est_public v122"] - by_country["est_public v110"]
        by_country["d_mock_f05"] = by_country["mock F0.5 v122"] - by_country["mock F0.5 v110"]
        display(by_country.round(5))
    else:
        print("no per-country scores in metrics.json")

### 5.3 What each layer adds

Stage 1 alone (its p1 through the threshold grid, tuned on the tune entities) carries switches
1–4, which all change stage 1 or its inputs; the two-stage rows add switches 5 and 6 on top,
and seed 42 alone against the mean of the 3 seeds isolates the averaging (each row's rule is
tuned for it). Deltas are against v110's two-stage est_public.

In [ ]:
if finished():
    seeds = v122.get("stage2_seeds", [])
    ladder = pd.Series({
        "v110 stage 1 alone": v110.get("single_stage_est_public"),
        "v110 two-stage": v110.get("est_public"),
        "v122 stage 1 alone": v122.get("single_stage_est_public"),
        f"v122 two-stage, seed {seeds[0] if seeds else '?'} alone":
            v122.get("one_seed_est_public"),
        f"v122 two-stage, mean of {len(seeds)} seeds": v122.get("est_public"),
    }, dtype=float, name="est_public").to_frame()
    ladder["delta_vs_v110"] = ladder["est_public"] - EST_TO_BEAT
    display(ladder.round(5))

### 5.4 What stage 2 reads

The extras' share of stage-2 gain (mean over parts and seeds), logged as `extras_gain_share`
(v044's interactions + missing_flags: 0.00084), and the 25 features with the most gain, with
the columns of the groups v122 adds marked. The column lists are `features.FEATURE_COLUMNS`'s,
copied here so the notebook imports nothing but `config`.

In [ ]:
NEW_COLUMNS = {  # features.FEATURE_COLUMNS of the groups v122 adds
    "nofill (stage 1)": ["nofill_ratio", "nofill_token_set", "nofill_jaccard", "nofill_eq",
                         "fill_n_l", "fill_n_r"],
    "tok_evidence (stage 1)": [f"te_{side}_{stat}" for side in ("pool", "s1")
                               for stat in ("sum", "min", "max", "decoy", "filler", "unknown")],
    "interactions (extra)": ["name_strong_addr_weak", "addr_strong_name_weak", "both_strong"],
    "missing_flags (extra)": ["nums_empty_l", "nums_empty_r"],
    "phonetic (extra)": ["phonetic_exact_match", "phonetic_jaccard",
                         "phonetic_token_overlap_ratio", "addr_phonetic_exact_match",
                         "addr_phonetic_jaccard", "addr_phonetic_token_overlap_ratio"],
}
GROUP_OF = {c: g for g, cols in NEW_COLUMNS.items() for c in cols}
if finished():
    top = pd.Series(v122.get("top_importance", {}), dtype=float, name="gain share")
    group = top.index.map(lambda c: GROUP_OF.get(c, "v110's columns"))
    print(f"extras' share of stage-2 gain: {v122.get('extras_gain_share', float('nan')):.5f}")
    print("gain share within the top 25, by group:",
          top.groupby(group).sum().round(5).to_dict())
    display(top.to_frame().assign(group=group).round(5))

## 6. Error analysis

Error counts on the mock val entities under the chosen rule (`evaluate.error_samples`, as in
v110), all in pairs: `false_merge` a predicted pair that is not true, of an entity with true
matches (the most expensive under the tight score); `missed` a true pair not predicted while its
entity predicted something; `false_singleton` the true pairs of an entity predicted empty;
`singleton_merge` the predicted pairs of a true singleton. Then where the val true pairs were
lost: never kept by blocking + the stage-1 filter (1 − candidate recall), or kept but not
predicted by stage 2 and the rule (candidate recall − pair recall). The script keeps counts
only; the scored pairs are in `artifacts/mock_scored.parquet`, but pairing them with the truth
needs the mock fold rebuilt, a job for a follow-up rather than this read-only notebook.

In [ ]:
KINDS = ["false_merge", "missed", "false_singleton", "singleton_merge"]
if finished():
    errors = pd.DataFrame({"v110": pd.Series(v110.get("errors_mock", {}), dtype=float),
                           "v122": pd.Series(v122.get("errors_mock", {}), dtype=float)}
                          ).reindex(KINDS)
    errors["delta"] = errors["v122"] - errors["v110"]
    errors["change"] = (errors["delta"] / errors["v110"]).map("{:+.1%}".format)
    counts = ["v110", "v122", "delta"]
    errors[counts] = errors[counts].astype("Int64")   # pair counts (<NA> where not logged)
    display(errors)
    nan = float("nan")
    lost = pd.DataFrame({name: {
        "lost before stage 2 (1 - candidate recall)": 1 - rec.get("cand_recall_val", nan),
        "lost at stage 2 + rule (candidate recall - pair recall)":
            rec.get("cand_recall_val", nan) - rec.get("pair_recall", nan),
        "found (pair recall)": rec.get("pair_recall", nan),
    } for name, rec in (("v110", v110), ("v122", v122))})
    lost["delta"] = lost["v122"] - lost["v110"]
    display(lost.round(5))

## 7. Log the result

The script logged the result itself, at its end (`tracking.log_result`: `metrics.json` in this
folder and the `v122` row of `experiments/experiments.csv`, stamped with the commit of `src/`),
with group `INT`, owner `M3`, parent `v110` and the user's decision rule: **KEEP if est_public >
v110's 0.97313, else DROP**. Nothing is logged from this notebook: the cell shows the logged row,
checks the decision against the rule, and lists the step times and the peak memory.

In [ ]:
if finished():
    row = {k: v for k, v in run.items() if k != "metrics"}
    display(pd.Series(row, name="experiments.csv row", dtype=object).to_frame())
    expected = "KEEP" if float(v122.get("est_public", float("nan"))) > EST_TO_BEAT else "DROP"
    print(f"decision logged {run.get('decision')!r}; the rule gives {expected!r}"
          + ("" if run.get("decision") == expected else "  <-- MISMATCH"))
    seconds = pd.Series({k.removesuffix("_seconds"): v for k, v in v122.items()
                         if k.endswith("_seconds")}, dtype=float, name="seconds")
    display(seconds.to_frame().assign(minutes=seconds / 60).round(1))
    print(f"peak RSS {v122.get('peak_rss_gb')} GB")

## 8. Conclusion

> **(filled in after the run)**: written from the numbers above once the script has logged its
> result (`metrics.json`). Every bullet below is a placeholder until then.

* **Result vs parent (§5.1–5.2):** est_public 0.97313 (v110) → … (Δ …); mock F0.5 0.98165 → …;
  India …, US … *(filled in after the run)*
* **Decision (§7; the user's rule: KEEP if est_public > 0.97313, else DROP):** …
  *(filled in after the run)*
* **Where the change comes from (§5.3, §5.4, §6):** stage 1 alone …; seed averaging …; the
  extras' gain share …; false merges and misses … *(filled in after the run)*
* **Next experiment:** … *(filled in after the run)*

## 9. Test inference (KEEP only)

The script writes the test files only for a KEEP (or with `--force-test`): `run_test_two_stage`
with this stage 1 and stage 2 (stage-1 outputs of the test partitions cached per country) into
`submissions/v122/`, our checker (`python -m entity_resolution.submission --check-ids`) on them,
candidates, matched share and matches per S1 by country (France included: every test S1 gets a
row), and for a KEEP a copy to `output/`, checked again. The organisers' validator is not on
this machine: M1 runs it before an upload. For a DROP the log ends with the `DROP:` line and no
test file is written. The last 60 lines of the run log follow; while the run is going, this
cell is the progress view.

In [ ]:
text = read_log()
if text is None:
    print("run.log not found")
else:
    if log_entries("done: total"):
        state = "finished, test files written"
    elif log_entries("DROP:"):
        state = "finished: DROP, no test files"
    elif log_entries("logged {"):
        state = "result logged; test inference running (or stopped: see the tail)"
    else:
        state = "running (or stopped: a traceback would end the tail)"
    print(f"run state: {state}\n" + "-" * 100)
    print("\n".join(text.rstrip().splitlines()[-60:]))